In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"]
# ...

# 1. Provide a list of multiple image paths
images = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104520.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104580.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104640.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104700.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104760.png",

]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions1 = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries1 = []
camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    points_cam = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_cam)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries1.append(pcd1)
    geometries1.append(camera_frame)

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries1.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries1:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
o3d.visualization.draw_geometries(geometries1)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [10]:
print(views[0]['img'].shape)

torch.Size([1, 3, 392, 518])


In [2]:
last_pose

array([[ 0.9993263 , -0.016044  ,  0.03300931, -0.6793342 ],
       [ 0.01432533,  0.998562  ,  0.05165954,  0.7274399 ],
       [-0.03379066, -0.05115186,  0.99811906,  1.6686119 ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]],
      dtype=float32)

In [12]:
from PIL import Image
import numpy as np
import torch
from mapanything.models import MapAnything
device = "cuda" if torch.cuda.is_available() else "cpu"
batch2 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104760.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104820.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104880.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104940.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105000.png",

]

last_pose = np.array([
    [ 0.9993263 , -0.016044  ,  0.03300931, -0.6793342 ],
    [ 0.01432533,  0.998562  ,  0.05165954,  0.7274399 ],
    [-0.03379066, -0.05115186,  0.99811906,  1.6686119 ],
    [ 0.        ,  0.        ,  0.        ,  1.        ]
])
view2_1 = {
    "img": np.array(Image.open(batch2[0]).convert("RGB")),
    'camera_poses': last_pose,
}


views2 = []
views2.append(view2_1)

for i, rgb in enumerate(batch2):
    if i > 0:
        views2.append({"img":np.array(Image.open(batch2[i]).convert("RGB"))})


In [14]:
view2_1['img'].shape

(300, 400, 3)

In [13]:
from mapanything.utils.image import preprocess_inputs
processed_views2 = preprocess_inputs(views2)

In [14]:
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
predictions2 = model.infer(
    processed_views2,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    ignore_calibration_inputs=False,
    ignore_depth_inputs=False,
    ignore_pose_inputs=False,
    ignore_depth_scale_inputs=False,
    ignore_pose_scale_inputs=False,
)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [28]:
geometries2 = []
camera_positions2 = []
last2_pose = None
for i, pred2 in enumerate(predictions2):
    
    points2 = pred2["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors2 = pred2["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd2 = o3d.geometry.PointCloud()
    pcd2.points = o3d.utility.Vector3dVector(points2)
    pcd2.colors = o3d.utility.Vector3dVector(colors2)

    pcd2.transform(last_pose)
    
    
    camera_pose2 = pred2["camera_poses"].squeeze().cpu().numpy()
    
    camera_pose2 = camera_pose2@last_pose
    
    # pcd1.transform(camera_pose)

    camera_center2 = camera_pose2[:3, 3]
    camera_positions2.append(camera_center2)
    
    
    camera_frame2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame2.transform(camera_pose2)
    
    
    
    geometries2.append(pcd2)
    geometries2.append(camera_frame2)

    if i == len(predictions2) - 1:
        last2_pose = camera_pose2

   

if len(camera_positions2) > 1:
    # Define the points for the line set
    line_points2 = o3d.utility.Vector3dVector(camera_positions2)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices2 = [[i, i + 1] for i in range(len(camera_positions2) - 1)]
    lines2 = o3d.utility.Vector2iVector(line_indices2)
    
    # Create the LineSet object
    camera_path2 = o3d.geometry.LineSet(points=line_points2, lines=lines2)
    
    # Set the color of the path to red
    camera_path2.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries2.append(camera_path2)


for geometry in geometries2:
    geometry.transform(transform_matrix)
    

In [24]:
o3d.visualization.draw_geometries(geometries2)

In [29]:
combined_geometries = geometries1 + geometries2


In [30]:
o3d.visualization.draw_geometries(
    combined_geometries,
    width=1280,
    height=720
)

In [31]:
last2_pose

array([[ 0.80566981,  0.01733574, -0.59211122, -1.76334827],
       [-0.14329199,  0.97558955, -0.16641037, -1.6413725 ],
       [ 0.57477268,  0.21891661,  0.7884871 ,  2.46478545],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [43]:
from PIL import Image
import numpy as np
import torch
from mapanything.models import MapAnything
device = "cuda" if torch.cuda.is_available() else "cpu"
batch3 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105000.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105060.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105120.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105180.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105240.png",

]



view3_1 = {
    "img": np.array(Image.open(batch3[0]).convert("RGB")),
    'camera_poses': last2_pose,
}


views3 = []
views3.append(view3_1)

for i, rgb in enumerate(batch3):
    if i > 0:
        views3.append({"img":np.array(Image.open(batch3[i]).convert("RGB"))})


In [44]:
processed_views3 = preprocess_inputs(views3)
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
predictions3 = model.infer(
    processed_views3,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    ignore_calibration_inputs=False,
    ignore_depth_inputs=False,
    ignore_pose_inputs=False,
    ignore_depth_scale_inputs=False,
    ignore_pose_scale_inputs=False,
)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [45]:
geometries3 = []
camera_positions3 = []
last3_pose = None
for i, pred3 in enumerate(predictions3):
    
    points3 = pred3["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors3 = pred3["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd3 = o3d.geometry.PointCloud()
    pcd3.points = o3d.utility.Vector3dVector(points3)
    pcd3.colors = o3d.utility.Vector3dVector(colors3)

    pcd3.transform(last2_pose)
    
    
    camera_pose3 = pred3["camera_poses"].squeeze().cpu().numpy()
    
    camera_pose3 = camera_pose3@last2_pose
    
    # pcd1.transform(camera_pose)

    camera_center3 = camera_pose3[:3, 3]
    camera_positions3.append(camera_center3)
    
    
    camera_frame3 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame3.transform(camera_pose3)
    
    
    
    geometries3.append(pcd3)
    geometries3.append(camera_frame3)

    if i == len(predictions3) - 1:
        last3_pose = camera_pose3

   

if len(camera_positions3) > 1:
    # Define the points for the line set
    line_points3 = o3d.utility.Vector3dVector(camera_positions3)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices3 = [[i, i + 1] for i in range(len(camera_positions3) - 1)]
    lines3 = o3d.utility.Vector2iVector(line_indices3)
    
    # Create the LineSet object
    camera_path3 = o3d.geometry.LineSet(points=line_points3, lines=lines3)
    
    # Set the color of the path to red
    camera_path3.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries3.append(camera_path3)


for geometry in geometries3:
    geometry.transform(transform_matrix)
    

In [35]:
o3d.visualization.draw_geometries(geometries3)

In [82]:
combined_geometries =geometries1 + geometries2  + geometries3

In [83]:
o3d.visualization.draw_geometries(combined_geometries)

In [52]:
last3_pose

array([[ 0.96541922,  0.11659607,  0.23317629, 10.20787562],
       [-0.06284766,  0.97212408, -0.22588706, -3.47038843],
       [-0.25301382,  0.20342113,  0.94583502, -3.79456264],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [70]:
from PIL import Image
import numpy as np
import torch
from mapanything.models import MapAnything
device = "cuda" if torch.cuda.is_available() else "cpu"
batch4 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105240.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105300.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105360.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105420.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106000.png"

]



view4_1 = {
    "img": np.array(Image.open(batch4[0]).convert("RGB")),
    'camera_poses': last3_pose,
}


views4 = []
views4.append(view4_1)

for i, rgb in enumerate(batch4):
    if i > 0:
        views4.append({"img":np.array(Image.open(batch4[i]).convert("RGB"))})


In [71]:
processed_views4 = preprocess_inputs(views4)
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
predictions4 = model.infer(
    processed_views4,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    ignore_calibration_inputs=False,
    ignore_depth_inputs=False,
    ignore_pose_inputs=False,
    ignore_depth_scale_inputs=False,
    ignore_pose_scale_inputs=False,
)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [78]:
# geometries4 = []
# camera_positions4 = []
# last4_pose = None
# for i, pred4 in enumerate(predictions4):
    
#     points4 = pred4["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
#     colors4 = pred4["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
#     pcd4 = o3d.geometry.PointCloud()
#     pcd4.points = o3d.utility.Vector3dVector(points4)
#     pcd4.colors = o3d.utility.Vector3dVector(colors4)

#     pcd4.transform(last3_pose)
     
    
#     camera_pose4 = pred4["camera_poses"].squeeze().cpu().numpy()
    
#     camera_pose4 = camera_pose4@last3_pose
    
#     # pcd1.transform(camera_pose)

#     camera_center4 = camera_pose4[:3, 3]
#     camera_positions4.append(camera_center4)
    
    
#     camera_frame4 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
#     camera_frame4.transform(camera_pose4)
    
    
    
#     geometries4.append(pcd4)
#     geometries4.append(camera_frame4)

#     if i == len(predictions4) - 1:
#         last4_pose = camera_pose4
geometries4 = []
camera_positions4 = []
last4_pose = None
for i, pred4 in enumerate(predictions4):
    
    # 1. FIX: Use local points from the camera's perspective
    points4 = pred4["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    colors4 = pred4["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd4 = o3d.geometry.PointCloud()
    pcd4.points = o3d.utility.Vector3dVector(points4)
    pcd4.colors = o3d.utility.Vector3dVector(colors4)
     
    # 2. FIX: The camera_pose4 from the model is ALREADY the correct global pose. Do not modify it.
    camera_pose4 = pred4["camera_poses"].squeeze().cpu().numpy()
    
    # 3. FIX: Transform the local points into the global world using their own correct global pose
    pcd4.transform(camera_pose4)
    
    camera_center4 = camera_pose4[:3, 3]
    camera_positions4.append(camera_center4)
    
    camera_frame4 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame4.transform(camera_pose4)
    
    geometries4.append(pcd4)
    geometries4.append(camera_frame4)

    if i == len(predictions4) - 1:
        # Save the final GLOBAL pose for the next batch
        last4_pose = pred4["camera_poses"].squeeze().cpu().numpy()


   

if len(camera_positions4) > 1:
    # Define the points for the line set
    line_points4 = o3d.utility.Vector3dVector(camera_positions4)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices4 = [[i, i + 1] for i in range(len(camera_positions4) - 1)]
    lines4 = o3d.utility.Vector2iVector(line_indices4)
    
    # Create the LineSet object
    camera_path4 = o3d.geometry.LineSet(points=line_points4, lines=lines4)
    
    # Set the color of the path to red
    camera_path4.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries4.append(camera_path4)


for geometry in geometries4:
    geometry.transform(transform_matrix)
    

In [79]:
o3d.visualization.draw_geometries(geometries4)

In [80]:
combined_geometries = geometries3 + geometries4

In [81]:
o3d.visualization.draw_geometries(combined_geometries)